# `dbspend360_cluster_spends_app`

Per-cluster spend rollup for **shared / interactive clusters**
(`cluster_source IN ('UI','API')`). Sibling pipeline to
`dbspend360_dbu_cost_app` + `databricks_job_spends_app`, but scoped to
non-job DBU usage where `usage_metadata.cluster_id IS NOT NULL AND
usage_metadata.job_run_id IS NULL`.

Joins:
- `system.compute.clusters` (deduped to one row per cluster) — for owner / name /
  security mode snapshot. Without dedup the slowly-changing snapshot would fan
  out cost rows by N history rows per cluster.
- `system.billing.usage` × `system.billing.list_prices` — DBU cost.
- `dbspend360_cloud_cost_explorer` — cloud VM / EBS / network cost
  (already keyed on `cluster_id`).

Merge key: `(cluster_id, workspace_id, usage_date)`.

Widgets mirror the DBU notebook: `catalog`, `schema`, `overlap_days`, `workspace_ids`.

In [ ]:
%run ./utils_common

In [ ]:
logger = setup_logger('ClusterSpendsReporter')

In [ ]:
dbutils.widgets.text('catalog', '', 'CATALOG')
dbutils.widgets.text('schema', '', 'SCHEMA')
dbutils.widgets.text('overlap_days', '3', 'Overlap days (min 2)')
dbutils.widgets.text('workspace_ids', '', 'Workspace IDs (comma-separated, blank=all)')

In [ ]:
# =======================================================
# Cluster Spends Client
# =======================================================
class ClusterSpendsClient:
  TABLE_NAME = 'dbspend360_cluster_spends'
  INTERACTIVE_SOURCES = ('UI', 'API')

  def __init__(
    self,
    audit_table: str,
    cloud_cost_table: str,
    target_table: str,
    error_log_table: str,
    overlap_days: int,
    logger=None,
  ):
    self.audit_table = audit_table
    self.cloud_cost_table = cloud_cost_table
    self.target_table = target_table
    self.error_log_table = error_log_table
    self.overlap_days = overlap_days
    self.logger = logger or logging.getLogger('ClusterSpendsClient')

    raw_ws = dbutils.widgets.get('workspace_ids')
    if raw_ws.strip() == '':
      self.workspace_ids = None
    else:
      self.workspace_ids = [w.strip() for w in raw_ws.split(',') if w.strip()]

  def _deduped_clusters_df(self):
    """One row per cluster_id, latest config snapshot.

    `system.compute.clusters` is a slowly-changing snapshot table (one row
    per config change). Joining the raw table fans out per-day costs by N
    history rows. We collapse to the latest config per cluster_id by
    change_time before any join.
    """
    win = spark.table('system.compute.clusters').filter(
      F.col('cluster_source').isin(list(self.INTERACTIVE_SOURCES))
    )
    if self.workspace_ids is not None:
      win = win.filter(F.col('workspace_id').isin(self.workspace_ids))

    from pyspark.sql.window import Window

    w = Window.partitionBy('cluster_id').orderBy(F.col('change_time').desc())
    return (
      win.withColumn('_rn', F.row_number().over(w))
      .filter(F.col('_rn') == 1)
      .drop('_rn')
      .select(
        'cluster_id',
        'workspace_id',
        'cluster_source',
        'cluster_name',
        'owned_by',
        'data_security_mode',
      )
    )

  def _dbu_costs_df(self, start_dt, end_dt):
    """Per-(cluster_id, workspace_id, usage_date) DBU $ for shared-cluster usage.

    Filters `system.billing.usage` to rows where:
    - `usage_metadata.cluster_id IS NOT NULL` (cluster-attributed usage)
    - `usage_metadata.job_run_id IS NULL`     (exclude job runs)
    """
    usage_df = (
      spark.table('system.billing.usage')
      .alias('usage')
      .filter(
        (F.col('usage.usage_date') >= F.lit(start_dt))
        & (F.col('usage.usage_date') <= F.lit(end_dt))
      )
      .filter(F.col('usage.usage_metadata')['cluster_id'].isNotNull())
      .filter(F.col('usage.usage_metadata')['job_run_id'].isNull())
    )
    if self.workspace_ids is not None:
      usage_df = usage_df.filter(F.col('usage.workspace_id').isin(self.workspace_ids))

    list_prices_df = spark.table('system.billing.list_prices').alias('list_prices')

    priced = usage_df.join(
      list_prices_df,
      on=(
        (F.col('usage.sku_name') == F.col('list_prices.sku_name'))
        & (F.col('usage.usage_start_time') >= F.col('list_prices.price_start_time'))
        & (
          (F.col('usage.usage_start_time') < F.col('list_prices.price_end_time'))
          | F.col('list_prices.price_end_time').isNull()
        )
      ),
      how='left',
    )

    return priced.groupBy(
      F.col('usage.usage_metadata')['cluster_id'].alias('cluster_id'),
      F.col('usage.workspace_id').alias('workspace_id'),
      F.col('usage.usage_date').alias('usage_date'),
    ).agg(
      F.sum(
        F.col('usage.usage_quantity') * F.col('list_prices.pricing')['default'].cast('double')
      ).alias('databricks_cost'),
      F.lit('USD').alias('currency'),
    )

  def compute_and_merge_cluster_spends(self):
    start_dt = end_dt = datetime.now(timezone.utc).date()
    try:
      start_dt, end_dt = get_date_window(self.audit_table, self.TABLE_NAME, self.overlap_days)

      valid, msg = validate_date_window(start_dt, end_dt)
      if not valid:
        raise DataQualityError(msg)

      self.logger.info(f'Building dbspend360_cluster_spends for {start_dt} \u2192 {end_dt}')

      ensure_cost_columns(self.target_table, logger=self.logger)

      clusters_df = self._deduped_clusters_df()
      dbu_df = self._dbu_costs_df(start_dt, end_dt)

      if dbu_df.limit(1).count() == 0:
        self.logger.info('No shared-cluster DBU rows in this date window; nothing to merge.')
        log_audit_run(
          self.audit_table,
          self.TABLE_NAME,
          start_dt,
          end_dt,
          'SUCCESS',
          0,
          'No shared-cluster DBU data in window',
        )
        return

      cloud_df = (
        spark.table(self.cloud_cost_table)
        .alias('cc')
        .filter(
          (F.col('cost_incurred_date') >= F.lit(start_dt))
          & (F.col('cost_incurred_date') <= F.lit(end_dt))
        )
      )

      cc_columns = {c.name for c in spark.table(self.cloud_cost_table).schema}
      has_segmented = 'compute_cost' in cc_columns
      has_other = 'other_cost' in cc_columns

      joined = (
        dbu_df.alias('d')
        .join(
          clusters_df.alias('c'),
          on=(F.col('d.cluster_id') == F.col('c.cluster_id')),
          how='inner',
        )
        .join(
          cloud_df,
          on=(
            (F.col('d.cluster_id') == F.col('cc.cluster_id'))
            & (F.col('d.usage_date') == F.col('cc.cost_incurred_date'))
          ),
          how='left',
        )
      )

      select_cols = [
        F.col('d.cluster_id').alias('cluster_id'),
        F.col('d.workspace_id').alias('workspace_id'),
        F.col('c.cluster_source').alias('cluster_source'),
        F.col('c.cluster_name').alias('cluster_name'),
        F.col('c.owned_by').alias('owned_by'),
        F.col('c.data_security_mode').alias('data_security_mode'),
        F.col('d.usage_date').alias('usage_date'),
        F.coalesce(F.col('cc.cloud_cost'), F.lit(0.0)).alias('cloud_cost'),
        F.col('d.databricks_cost').alias('databricks_cost'),
        F.coalesce(F.col('cc.currency'), F.col('d.currency'), F.lit('USD')).alias('currency'),
      ]

      if has_segmented:
        select_cols.extend(
          [
            F.col('cc.compute_cost').alias('compute_cost'),
            F.col('cc.storage_cost').alias('storage_cost'),
            F.col('cc.network_cost').alias('network_cost'),
          ]
        )
      if has_other:
        select_cols.append(F.col('cc.other_cost').alias('other_cost'))

      final_df = joined.select(*select_cols)

      if not has_segmented:
        final_df = (
          final_df.withColumn('compute_cost', F.lit(None).cast('double'))
          .withColumn('storage_cost', F.lit(None).cast('double'))
          .withColumn('network_cost', F.lit(None).cast('double'))
        )
      if not has_other:
        final_df = final_df.withColumn('other_cost', F.lit(None).cast('double'))

      final_df = (
        final_df.withColumn(
          'total_cost',
          F.coalesce(F.col('cloud_cost'), F.lit(0.0))
          + F.coalesce(F.col('databricks_cost'), F.lit(0.0)),
        )
        .withColumn('created_at', F.current_timestamp())
        .withColumn('updated_at', F.current_timestamp())
      )
      final_df = safe_cache(final_df)

      row_count = final_df.count()

      validate_source_schema(
        final_df,
        {
          'cluster_id': 'string',
          'workspace_id': 'string',
          'usage_date': 'date',
          'cloud_cost': 'double',
          'databricks_cost': 'double',
        },
        self.target_table,
        self.logger,
      )
      validate_no_negative_costs(
        final_df,
        [
          'cloud_cost',
          'databricks_cost',
          'total_cost',
          'compute_cost',
          'storage_cost',
          'network_cost',
          'other_cost',
        ],
        self.target_table,
        self.logger,
      )
      validate_currency_consistency(final_df, 'currency', self.target_table, self.logger)

      target = DeltaTable.forName(spark, self.target_table)
      (
        target.alias('t')
        .merge(
          final_df.alias('s'),
          't.cluster_id = s.cluster_id AND t.workspace_id = s.workspace_id '
          'AND t.usage_date = s.usage_date',
        )
        .whenMatchedUpdate(
          set={
            'cluster_source': 's.cluster_source',
            'cluster_name': 's.cluster_name',
            'owned_by': 's.owned_by',
            'data_security_mode': 's.data_security_mode',
            'cloud_cost': 's.cloud_cost',
            'compute_cost': 's.compute_cost',
            'storage_cost': 's.storage_cost',
            'network_cost': 's.network_cost',
            'other_cost': 's.other_cost',
            'databricks_cost': 's.databricks_cost',
            'currency': 's.currency',
            'total_cost': 's.total_cost',
            'updated_at': 'current_timestamp()',
          }
        )
        .whenNotMatchedInsert(
          values={
            'cluster_id': 's.cluster_id',
            'workspace_id': 's.workspace_id',
            'cluster_source': 's.cluster_source',
            'cluster_name': 's.cluster_name',
            'owned_by': 's.owned_by',
            'data_security_mode': 's.data_security_mode',
            'usage_date': 's.usage_date',
            'cloud_cost': 's.cloud_cost',
            'compute_cost': 's.compute_cost',
            'storage_cost': 's.storage_cost',
            'network_cost': 's.network_cost',
            'other_cost': 's.other_cost',
            'databricks_cost': 's.databricks_cost',
            'currency': 's.currency',
            'total_cost': 's.total_cost',
            'created_at': 'current_timestamp()',
            'updated_at': 'current_timestamp()',
          }
        )
        .execute()
      )

      safe_unpersist(final_df)
      get_merge_metrics(self.target_table, self.logger)

      validate_post_merge(
        self.target_table,
        'usage_date',
        start_dt,
        end_dt,
        row_count,
        self.logger,
      )

      log_audit_run(
        self.audit_table,
        self.TABLE_NAME,
        start_dt,
        end_dt,
        'SUCCESS',
        row_count,
        '',
      )
      self.logger.info(
        f'Merged {row_count} rows into {self.target_table} for {start_dt} \u2192 {end_dt}.'
      )

    except Exception as e:
      msg = str(e)[:1000]
      self.logger.error(f'Run failed: {msg}')
      try:
        log_audit_run(
          self.audit_table,
          self.TABLE_NAME,
          start_dt,
          end_dt,
          'FAILED',
          0,
          msg,
        )
      except Exception:
        self.logger.error('Failed to write FAILED audit entry')
      raise

In [ ]:
# =======================================================
# APP
# =======================================================
class ClusterSpendsReporterApp:
  def __init__(self):
    catalog = dbutils.widgets.get('catalog')
    schema = dbutils.widgets.get('schema')
    ov_days = get_overlap_days(dbutils.widgets.get('overlap_days'), logger=logger)

    self.client = ClusterSpendsClient(
      audit_table=build_table_fqn(catalog, schema, 'dbspend360_audit_log'),
      cloud_cost_table=build_table_fqn(catalog, schema, 'dbspend360_cloud_cost_explorer'),
      target_table=build_table_fqn(catalog, schema, 'dbspend360_cluster_spends'),
      error_log_table=build_table_fqn(catalog, schema, 'dbspend360_error_log'),
      overlap_days=ov_days,
      logger=logger,
    )

  def run(self):
    self.client.compute_and_merge_cluster_spends()

In [ ]:
# =======================================================
# Execute
# =======================================================
app = ClusterSpendsReporterApp()
app.run()